# The EM Algorithm

[← Back to wiki](https://ml-viz.vercel.app/wiki/em-algorithm)

> **To save your work:** click the **Copy to Drive** button at the top, or go to File → Save a copy in Drive. Changes to this view are not saved.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.stats import norm
plt.style.use('dark_background')

## From-scratch GMM EM (1D)

In [ ]:
def em_gmm(X, K=2, max_iter=100, tol=1e-6, seed=42):
    rng = np.random.default_rng(seed)
    idx = rng.choice(len(X), K, replace=False)
    mu = X[idx].copy()
    sigma = np.ones(K) * X.std()
    pi = np.ones(K) / K
    log_liks = []

    for it in range(max_iter):
        # E-step
        resp = np.array([pi[k] * norm.pdf(X, mu[k], sigma[k]) for k in range(K)]).T
        resp /= resp.sum(axis=1, keepdims=True)

        # M-step
        N = resp.sum(axis=0)
        mu = (resp * X[:,None]).sum(axis=0) / N
        sigma = np.sqrt(((resp * (X[:,None]-mu)**2).sum(axis=0)) / N)
        pi = N / len(X)

        ll = np.log(sum(pi[k]*norm.pdf(X,mu[k],sigma[k]) for k in range(K))+1e-300).sum()
        log_liks.append(ll)
        if it > 0 and abs(log_liks[-1]-log_liks[-2]) < tol:
            print(f"Converged at iteration {it}")
            break

    return mu, sigma, pi, resp, log_liks

# Two-component toy data
rng = np.random.default_rng(0)
X = np.concatenate([rng.normal(2, 1, 150), rng.normal(7, 1.5, 100)])
mu, sigma, pi, resp, lls = em_gmm(X, K=2)
print(f"mu={mu.round(2)}, sigma={sigma.round(2)}, pi={pi.round(2)}")

## Convergence and fit

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Log-likelihood curve
axes[0].plot(lls, color='#6366f1')
axes[0].set_xlabel('Iteration'); axes[0].set_ylabel('Log-likelihood')
axes[0].set_title('EM convergence')

# Final fit
xs = np.linspace(X.min()-1, X.max()+1, 400)
axes[1].hist(X, bins=40, density=True, alpha=0.4, color='#94a3b8')
for k in range(2):
    axes[1].plot(xs, pi[k]*norm.pdf(xs, mu[k], sigma[k]),
                 label=f'Component {k+1}: μ={mu[k]:.1f}, σ={sigma[k]:.1f}')
axes[1].set_title('Fitted GMM'); axes[1].legend()
plt.tight_layout(); plt.show()

## ✏️ Your turn

**Task:** Run EM with `K=3` on the same dataset. Which value of K gives the highest final log-likelihood?

**Extension:** Add a restart loop — run EM 5 times with different seeds and return the result with the highest log-likelihood.

In [ ]:
# TODO(you): run EM with K=3
# mu3, sigma3, pi3, resp3, lls3 = em_gmm(X, K=?)
# Compare lls[-1] vs lls3[-1]

In [ ]:
# Silent assert — uncomment once done
# assert len(lls3) > 0, 'EM should produce log-likelihoods'
# assert lls3[-1] > lls[-1] or True, 'K=3 may or may not improve fit'

<details><summary>Solution</summary>

```python
mu3, sigma3, pi3, resp3, lls3 = em_gmm(X, K=3)
print(f'K=2 final LL: {lls[-1]:.2f}')
print(f'K=3 final LL: {lls3[-1]:.2f}')
# K=3 should give a higher (less negative) log-likelihood on this data
```
</details>